## Gestión Dinámica de Centros de Datos para Optimizar Consumo Energético y Rendimiento

### Introducción
El crecimiento exponencial de la tecnología y la digitalización ha llevado a una proliferación de centros de datos (Data Centers), los cuales son esenciales para el almacenamiento y procesamiento de información a nivel global. Sin embargo, estos centros representan uno de los mayores consumidores de energía, contribuyendo significativamente al gasto eléctrico mundial y generando una importante huella de carbono.

A pesar de este alto consumo, los centros de datos poseen un potencial inexplorado para servir como herramienta de gestión de la demanda eléctrica. Este proyecto plantea la posibilidad de utilizar centros de datos como una fuente de flexibilidad durante periodos de tensión en el suministro eléctrico, permitiendo mitigar apagones mediante la reducción temporal de su consumo energético. Esta estrategia implica una disminución en el procesamiento de ciertas tareas a cambio de beneficios económicos, creando un balance entre eficiencia energética e ingresos monetarios.


## Objetivo del Proyecto
El objetivo principal de este trabajo es desarrollar un modelo de gestión de centros de datos que permita:
- Maximizar los ingresos al reducir el consumo durante periodos críticos, priorizando tareas con mayor rentabilidad.
- Minimizar los tiempos de espera y redirigir procesos a otros centros en la red, evaluando el coste de latencia.
- Equilibrar un enfoque multi-criterio que tome en cuenta tanto los beneficios económicos como el prestigio derivado de mantener tiempos de respuesta bajos.

A través de este modelo, se busca proporcionar una herramienta valiosa para las empresas, facilitando la toma de decisiones dinámica y adaptativa en entornos de alta demanda y volatilidad energética.


## Contexto y Relevancia
Según estudios recientes, los centros de datos representan aproximadamente el 1% del consumo eléctrico global, con una tendencia creciente debido a la demanda continua de servicios digitales. En algunos países, durante picos de consumo eléctrico, los centros de datos pueden ser llamados a reducir su carga, pero esta práctica aún es poco explotada de manera sistemática.

Este proyecto ofrece una solución integral para:
1. Aumentar la sostenibilidad de los centros de datos.
2. Proporcionar ingresos adicionales a través de la participación en mercados de gestión de demanda.
3. Contribuir a la estabilidad de las redes eléctricas.


## Modelo de Gestión y Redirección de Procesos
El modelo conceptual del proyecto considera al centro de datos como una entidad con entradas (energía consumida) y salidas (procesos ejecutados). En situaciones de emergencia eléctrica, el centro de datos puede reducir su consumo a través de dos estrategias principales:

- **Desaceleración de procesos**: Disminuir la carga de trabajo, postergando o reduciendo la cantidad de tareas ejecutadas.
- **Redirección de procesos**: Transferir procesos a otros centros de datos de la red, con la penalización de una mayor latencia y costes adicionales.

Los procesos se clasifican según su prioridad:
- **Alta prioridad**: Procesos que pagan más para ser ejecutados rápidamente, sin importar las condiciones energéticas.
- **Baja prioridad**: Procesos que pueden esperar o ser redirigidos.


## Conclusiones y Futuras Líneas de Trabajo
Este proyecto plantea una innovadora estrategia para aprovechar los centros de datos en la gestión de crisis energéticas, abriendo nuevas vías de ingresos y optimización. Las futuras líneas de investigación podrán enfocarse en:
- **Automatización de la toma de decisiones** mediante algoritmos de inteligencia artificial.
- **Integración con redes eléctricas inteligentes** para mejorar la respuesta en tiempo real.
- **Análisis multicriterio más avanzado** que permita considerar diferentes escenarios y restricciones.

Con este enfoque, se espera que los centros de datos se conviertan en una pieza clave para garantizar la estabilidad y sostenibilidad del suministro eléctrico global en el futuro.


### Importing packages and modules

In [270]:
!pip install pyomo


[notice] A new release of pip is available: 24.0 -> 24.3.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [271]:
# module for building the pyomo model
import pyomo.environ as pe
# module for solving the pyomo model
import pyomo.opt as po
import random

### Create the model

In [272]:
model = pe.ConcreteModel()

Order to build the model:
1. Sets
1. Parameters
1. Variables
1. Objective function
1. Constraints

---------------------------------

#### Sets

$i$: nodes of the data center {1, .... 10}

$j$: tasks {1, .... 80}

In [273]:
# Conjuntos
model.i = pe.Set(initialize=[i for i in range(1, 11)])  # Nodos del centro de datos {1, ..., 10}
model.j = pe.Set(initialize=[j for j in range(1, 81)])  # Tareas {1, ..., 80}


---------------------------------

#### Parameters

$C{_i}$: Maximum capacity of node $i$ [Mb]

In [274]:
max_capacity = {
    1: 190, 2: 240, 3: 150, 4: 210, 5: 140,
    6: 180, 7: 170, 8: 220, 9: 180, 10: 190
}
model.max_capacity = pe.Param(model.i, initialize=max_capacity)

$S{_j}$: Memory required by task $j$ [Mb]

In [275]:
memory_required = {
    j: 15 + j % 10 for j in model.j  # Valores entre 15 y 25
}
model.memory_required = pe.Param(model.j, initialize=memory_required)

$I_{i,j}$: If i is the initial node $i$ of task $j$  {$1,0$}

In [276]:
initial_node = {
    # Empiezan todos en el nodo 5 
    (i, j): 1 if i==5 else 0
    for i in model.i for j in model.j
}
model.initial_node = pe.Param(model.i, model.j, initialize=initial_node, within=pe.Binary)

$T{_j}$: Time to complete task $j$ [s]

In [277]:
task_time = {
    j: 50 + j * 2 for j in model.j  # Incrementa linealmente entre 50 y 210
}
model.task_time = pe.Param(model.j, initialize=task_time)

$D_{i,i'}$: Time to transport from node $i$ to $i'$ [s]

In [278]:
transport_time = {
    (i, i_prime): 10 + abs(i - i_prime)*5  if i != i_prime else 0
    for i in model.i for i_prime in model.i
}
model.transport_time = pe.Param(model.i, model.i, initialize=transport_time, default=0)

$P_{j}$: Priority group of task $j$ {$1,.....80$}

In [279]:
priority_group = {
    j: 1 if j<=15 else 2 if j<=40 else 3 for j in model.j
}
model.priority_group = pe.Param(model.j, initialize=priority_group)

$Q_{i}$: Quality of node $i$ {${1 .....10}$}



In [280]:
node_quality = {
    1: 8, 2: 10, 3: 2, 4: 3, 5: 5,
    6: 4, 7: 6, 8: 1, 9: 9, 10: 7
}

model.node_quality = pe.Param(model.i, initialize=node_quality)

$E$: % Energy decrease required [%]

In [281]:
energy_reduction = 15
model.energy_reduction = pe.Param(initialize=energy_reduction)

$R{_i}$: Time increase due to reduction in processing capacity  [s/%]

In [282]:
time_increase = {
    1: 18, 2: 12, 3: 28, 4: 30, 5: 14,
    6: 25, 7: 27, 8: 24, 9: 16, 10: 15
}
model.time_increase = pe.Param(model.i, initialize=time_increase)

---------------------------------

#### Variables

$X_{i,i',j}$: If task $j$ is transported from node $i$ to $i'$ {$0,1$}

In [283]:
model.X = pe.Var(model.i, model.i, model.j, within=pe.Binary)

$A_{i,j}$: If task $j$ is assigned to be completed in node $i$ {$0,1$}

In [284]:
model.A = pe.Var(model.i, model.j, within=pe.Binary)

$Y{_i}$: % of energy reduced on node $i$

In [285]:
model.Y = pe.Var(model.i, within=pe.NonNegativeReals, bounds=(0, 100))

$K{_i}$: Variable for extra time delay if $Y{_i} > 20$ 

In [286]:
model.K = pe.Var(model.i, within=pe.NonNegativeReals)

$Z$: Minimun satisfaction for tasks

In [287]:
model.Z = pe.Var(within=pe.NonNegativeReals)

$EX_{i}$: Auxiliar binary variable that indicates if $Y{_i} > 20$ {$0,1$}

In [288]:
model.EX = pe.Var(model.i, within=pe.Binary)

$M_{j,j'}$: If task $j$ has priority over task $j'$ {$0,1$}

In [289]:
model.M = pe.Var(model.j, model.j, within=pe.Binary)

---------------------------------

#### Objective Function

a)
$$
\min \sum_{i, i', j} \left( T_j + X_{i,i',j} \cdot D_{i,i'} + \left( Y_{i'} + K_{i'}  \right)\cdot R_i \right)
$$



In [290]:
# Definición de la función objetivo en Pyomo
def obj_rule_time(model):
    return sum(
        model.task_time[j] + 
        model.X[i, i_prime, j] * model.transport_time[i, i_prime] + 
        (model.Y[i_prime] + model.K[i_prime]) * model.time_increase[i]
        for i in model.i
        for i_prime in model.i
        for j in model.j
    )

# Agregar la función objetivo al modelo
model.total_time = pe.Objective(rule=obj_rule_time, sense=pe.minimize)


b)
$$
\max Z
$$

In [291]:
# Definición de la función objetivo en Pyomo
def obj_rule_satisfaction(model):
    return model.Z

# Agregar la función objetivo al modelo
model.max_satisfaction = pe.Objective(rule=obj_rule_satisfaction, sense=pe.maximize)
model.max_satisfaction.deactivate()

---------------------------------

#### Constraints 

**1. Each task is transported only once**  
   $$
   \sum_{i,i'} x_{ii'j} \leq 1 \quad \forall j
   
   $$

In [292]:
model.eq_transport_once = pe.ConstraintList()

for j in model.j:
    model.eq_transport_once.add(
        sum(model.X[i, i_prime, j] for i in model.i for i_prime in model.i) <= 1
    )


**2. Each task is assigned to a single node**  
   $$
   \sum_{i} A_{ij} = 1 \quad \forall j
   $$

In [293]:
model.eq_task_assignment = pe.ConstraintList()

for j in model.j:
    model.eq_task_assignment.add(
        sum(model.A[i, j] for i in model.i) == 1
    )


**3. Maximum processing capacity for each node**  
   $$
   \sum_{j} A_{ij} \cdot S_j \leq   C_i\cdot(100 - Y_i)/100  \quad \forall i
   $$

In [294]:
model.eq_max_capacity = pe.ConstraintList()

for i in model.i:
    model.eq_max_capacity.add(
        sum(model.A[i, j] * model.memory_required[j] for j in model.j) <= 
        ((100 - model.Y[i]) * model.max_capacity[i])/100
    )


**4. Assigning a task to a different node implies transportation**  
   $$
   I_{ij} \cdot A_{i'j} = X_{ii'j} \quad \forall i, i', j \; i \neq i'
   $$

In [295]:
model.eq_transport_assignment = pe.ConstraintList()

for i in model.i:
    for i_prime in model.i:
        if i != i_prime:
            for j in model.j:
                model.eq_transport_assignment.add(
                    model.initial_node[i, j] * model.A[i_prime, j] == model.X[i, i_prime, j]
                )


*5. More important tasks are assigned to better nodes*

- If $ P_{j'} - P_j \geq 1 $, then:
$$
M_{jj'} = 1 \quad ; \quad P_{j'} - P_j \leq 1 - 1 + (M+1)M_{jj'}
$$

Additionally:
$$
P_{j'} - P_j \leq 3M_{jj'} \quad \forall j, j', j \neq j'
$$

- If $ M_{jj'} = 1 $, then:
$$
\sum_{i} (A_{ij'} - A_{ij})Q_i \geq 0 \quad ; \quad
\sum_{i} (A_{ij'} - A_{ij})Q_i \geq m(1 - M_{jj'})
$$
as
Where:
$$
10 \geq m \quad ; \quad m = 10
$$

- Finally, the exclusivity constraints:
$$
\sum_{i} (A_{ij'} - A_{ij})Q_i \geq -10 (1-M_{j,j´}) \quad \forall j, j', j \neq j' \quad ; \quad
M_{jj'} + M_{j'j} \leq 1
$$

In [296]:
model.eq_priority_diff = pe.ConstraintList()

for j in model.j:
    for j_prime in model.j:
        if j != j_prime:
            model.eq_priority_diff.add(
                model.priority_group[j_prime] - model.priority_group[j] <= 3 * model.M[j, j_prime]
            )


In [297]:
model.eq_node_quality = pe.ConstraintList()

m = 10  # Minimum threshold
for j in model.j:
    for j_prime in model.j:
        if j != j_prime:
            model.eq_node_quality.add(
                sum((model.A[i, j_prime] - model.A[i, j]) * model.node_quality[i] for i in model.i) >= 
                -m * (1 - model.M[j, j_prime])
            )


In [298]:
model.eq_exclusivity = pe.ConstraintList()

for j in model.j:
    for j_prime in model.j:
        if j != j_prime:
            model.eq_exclusivity.add(
                model.M[j, j_prime] + model.M[j_prime, j] <= 1
            )


**6. Energy decrease must be satisfied**

- The following constraint ensures energy decrease:
$$
\sum_{i} Y_i \cdot C_i \geq E \cdot \sum_{i} C_i
$$


In [299]:
model.eq_energy_decrease = pe.Constraint(
    expr=sum(model.Y[i] * model.max_capacity[i] for i in model.i) >= 
         model.energy_reduction * sum(model.max_capacity[i] for i in model.i)
)


**7. $Y_i$ is a percentage**

Constraints for $y_i$:
$$
Y_i \geq 0 \quad \forall i
$$
$$
Y_i \leq 100 \quad \forall i
$$

In [300]:
model.eq_y_percentage = pe.ConstraintList()

for i in model.i:
    model.eq_y_percentage.add(model.Y[i] >= 0)     
    model.eq_y_percentage.add(model.Y[i] <= 100)  


**8. Extra time if a node is under 80% of its capacity**

- If $ Y_i - 20 \geq 0 $; then:
$$
E X_i = 1 \quad ; \quad y_i - 20 \leq - 1 + (M+1)\cdot EX_i
$$
Where:
$$
80 \leq M \quad ; \quad M = 80
$$

Additional constraints:
$$
\frac{1}{81}(Y_i - 19) \leq EX_i \quad \forall i
$$

- If $ EX_i = 1 $, then:
$$
Y_i - K_i \leq 0
$$
$$
Y_i - K_i \leq M(1 - EX_i)
$$

Where:
$$
20 \leq M \quad ; \quad M = 20
$$

Finally:
$$
K_i \geq Y_i - 20(1 - EX_i) \quad \forall i
$$
$$
K_i \geq 0 \quad \forall i
$$

In [301]:
model.eq_ex_binary = pe.ConstraintList()

for i in model.i:
    model.eq_ex_binary.add(((model.Y[i] - 19) / 81) <= model.EX[i])          # \( \frac{1}{81}(Y_i - 19) \leq EX_i \)


In [302]:
model.eq_k_constraints = pe.ConstraintList()

for i in model.i:
    # \( K_i \geq Y_i - 20(1 - EX_i) \)
    model.eq_k_constraints.add(model.K[i] >= model.Y[i] - 20 * (1 - model.EX[i]))

    # \( K_i \geq 0 \)
    model.eq_k_constraints.add(model.K[i] >= 0)



**9. Measure minimum satisfaction**

$$
Z \leq \sum_{i} A_{ij} \cdot 1 / Q_i \cdot 1 / P_j \quad \forall j
$$

In [303]:
model.satisfaction = pe.ConstraintList()

for j in model.j:
    model.satisfaction.add(model.Z <= sum(model.A[i,j]*(1/model.priority_group[j])*(1/model.node_quality[i]) for i in model.i))

# Objective function 1: Minimizing time

### Solver

In [304]:
solver = pe.SolverFactory('gurobi')  # Puedes cambiar por cualquier solver que tengas instalado
results = solver.solve(model)

# Verifica que la solución sea óptima
if results.solver.termination_condition == pe.TerminationCondition.optimal:
    print("Modelo resuelto exitosamente")
else:
    print("Hubo un problema al resolver el modelo:", results.solver.termination_condition)


Modelo resuelto exitosamente


### Valor de la Función Objetivo

In [305]:
# valor de la funcion objetivo primera

total_time =  model.total_time()
print("Tiempo requerido por el centro de datos: {:.2f} horas".format(total_time/3600))



Tiempo requerido por el centro de datos: 943.79 horas


In [306]:
print("Tareas transportadas:")
for j in model.j:
    for i in model.i:
        for i_prime in model.i:
            if pe.value(model.X[i, i_prime, j]) == 1:
                print('La tarea {} fue transportada del nodo {} al nodo {} suponiendo un tiempo añadido de {} segundos'.format(j, i, i_prime, pe.value(model.transport_time[i,i_prime])))


Tareas transportadas:
La tarea 1 fue transportada del nodo 5 al nodo 3 suponiendo un tiempo añadido de 20 segundos
La tarea 2 fue transportada del nodo 5 al nodo 3 suponiendo un tiempo añadido de 20 segundos
La tarea 3 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 4 fue transportada del nodo 5 al nodo 3 suponiendo un tiempo añadido de 20 segundos
La tarea 5 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 6 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 7 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 8 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 9 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 10 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 11 fue transportada del nodo 5 al nodo

In [307]:
# Mostrar el porcentaje de reducción de energía en cada nodo
for i in model.i:
    y_value = pe.value(model.Y[i])
    print('El nodo {} tiene una reducción de energía del {:.2f} %'.format(i, y_value))


El nodo 1 tiene una reducción de energía del 18.95 %
El nodo 2 tiene una reducción de energía del 19.00 %
El nodo 3 tiene una reducción de energía del 0.00 %
El nodo 4 tiene una reducción de energía del 19.00 %
El nodo 5 tiene una reducción de energía del 0.00 %
El nodo 6 tiene una reducción de energía del 19.00 %
El nodo 7 tiene una reducción de energía del 7.53 %
El nodo 8 tiene una reducción de energía del 19.00 %
El nodo 9 tiene una reducción de energía del 19.00 %
El nodo 10 tiene una reducción de energía del 18.95 %


In [308]:
# Mostrar la memoria ocupada en cada nodo
for i in model.i:
    available_memory = (pe.value(model.max_capacity[i])*(100-pe.value(model.Y[i]))/100)
    used_memory = sum(pe.value(model.A[i, j]) * model.memory_required[j] for j in model.j)
    print("El nodo {} tiene ocupado {} Mb de los {:.2f} Mb disponibles, quedando {:.2f} Mb libres".format(i,used_memory, available_memory, available_memory-used_memory))

El nodo 1 tiene ocupado 154.0 Mb de los 154.00 Mb disponibles, quedando 0.00 Mb libres
El nodo 2 tiene ocupado 192.0 Mb de los 194.40 Mb disponibles, quedando 2.40 Mb libres
El nodo 3 tiene ocupado 148.0 Mb de los 150.00 Mb disponibles, quedando 2.00 Mb libres
El nodo 4 tiene ocupado 167.0 Mb de los 170.10 Mb disponibles, quedando 3.10 Mb libres
El nodo 5 tiene ocupado 130.0 Mb de los 140.00 Mb disponibles, quedando 10.00 Mb libres
El nodo 6 tiene ocupado 138.0 Mb de los 145.80 Mb disponibles, quedando 7.80 Mb libres
El nodo 7 tiene ocupado 154.0 Mb de los 157.20 Mb disponibles, quedando 3.20 Mb libres
El nodo 8 tiene ocupado 178.0 Mb de los 178.20 Mb disponibles, quedando 0.20 Mb libres
El nodo 9 tiene ocupado 145.0 Mb de los 145.80 Mb disponibles, quedando 0.80 Mb libres
El nodo 10 tiene ocupado 154.0 Mb de los 154.00 Mb disponibles, quedando 0.00 Mb libres


---------------------------------------------------------------------------------------------------------------------------------------------------

# Objective function 2: Maximizing minimum satisfaction 

### Solver

In [309]:
model.total_time.deactivate()
model.max_satisfaction.activate()
results = solver.solve(model)

# Verifica que la solución sea óptima
if results.solver.termination_condition == pe.TerminationCondition.optimal:
    print("Modelo resuelto exitosamente")
else:
    print("Hubo un problema al resolver el modelo:", results.solver.termination_condition)


Modelo resuelto exitosamente


### Valor de la Función Objetivo

In [310]:
# valor de la funcion objetivo primera
total_time =  model.total_time()
print(f"Minima satisfaccion de las tareas: {model.max_satisfaction()}")

print("Tiempo requerido por el centro de datos: {:.2f} horas".format(total_time/3600))


Minima satisfaccion de las tareas: 0.03703703714074122
Tiempo requerido por el centro de datos: 1585.06 horas


In [311]:
print("Tareas transportadas:")
for j in model.j:
    for i in model.i:
        for i_prime in model.i:
            if pe.value(model.X[i, i_prime, j]) == 1:
                print('La tarea {} fue transportada del nodo {} al nodo {} suponiendo un tiempo añadido de {} segundos'.format(j, i, i_prime, pe.value(model.transport_time[i,i_prime])))


Tareas transportadas:
La tarea 1 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 2 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 3 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 4 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 5 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 6 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 7 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 8 fue transportada del nodo 5 al nodo 3 suponiendo un tiempo añadido de 20 segundos
La tarea 9 fue transportada del nodo 5 al nodo 3 suponiendo un tiempo añadido de 20 segundos
La tarea 10 fue transportada del nodo 5 al nodo 8 suponiendo un tiempo añadido de 25 segundos
La tarea 11 fue transportada del nodo 5 al nodo

In [312]:
# Mostrar el porcentaje de reducción de energía en cada nodo
for i in model.i:
    y_value = pe.value(model.Y[i])
    print('El nodo {} tiene una reducción de energía del {:.2f} %'.format(i, y_value))


El nodo 1 tiene una reducción de energía del 4.21 %
El nodo 2 tiene una reducción de energía del 100.00 %
El nodo 3 tiene una reducción de energía del 0.67 %
El nodo 4 tiene una reducción de energía del 7.62 %
El nodo 5 tiene una reducción de energía del 0.71 %
El nodo 6 tiene una reducción de energía del 0.56 %
El nodo 7 tiene una reducción de energía del 7.65 %
El nodo 8 tiene una reducción de energía del 0.91 %
El nodo 9 tiene una reducción de energía del 8.33 %
El nodo 10 tiene una reducción de energía del 6.84 %


In [313]:
# Mostrar la memoria ocupada en cada nodo
for i in model.i:
    available_memory = (pe.value(model.max_capacity[i])*(100-pe.value(model.Y[i]))/100)
    used_memory = sum(pe.value(model.A[i, j]) * model.memory_required[j] for j in model.j)
    print("El nodo {} tiene ocupado {} Mb de los {:.2f} Mb disponibles, quedando {:.2f} Mb libres".format(i,used_memory, available_memory, available_memory-used_memory))

El nodo 1 tiene ocupado 182.0 Mb de los 182.00 Mb disponibles, quedando 0.00 Mb libres
El nodo 2 tiene ocupado 0.0 Mb de los 0.00 Mb disponibles, quedando 0.00 Mb libres
El nodo 3 tiene ocupado 149.0 Mb de los 149.00 Mb disponibles, quedando 0.00 Mb libres
El nodo 4 tiene ocupado 194.0 Mb de los 194.00 Mb disponibles, quedando 0.00 Mb libres
El nodo 5 tiene ocupado 139.0 Mb de los 139.00 Mb disponibles, quedando 0.00 Mb libres
El nodo 6 tiene ocupado 179.0 Mb de los 179.00 Mb disponibles, quedando 0.00 Mb libres
El nodo 7 tiene ocupado 157.0 Mb de los 157.00 Mb disponibles, quedando 0.00 Mb libres
El nodo 8 tiene ocupado 218.0 Mb de los 218.00 Mb disponibles, quedando 0.00 Mb libres
El nodo 9 tiene ocupado 165.0 Mb de los 165.00 Mb disponibles, quedando 0.00 Mb libres
El nodo 10 tiene ocupado 177.0 Mb de los 177.00 Mb disponibles, quedando 0.00 Mb libres
